In [1]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O tinyshakespeare.txt

In [2]:
from torch import nn

class Encoder:

    def __init__(self, path):
        with open(path) as f:
            lines = f.readlines()

        text = "\n".join(lines)

        self.dictionary = {}

        index = 0
        for char in text:
            if char in self.dictionary:
                continue

            self.dictionary[char] = index
            index += 1

    def encode(self, text):
        return [self.dictionary[char] for char in text]

    def decode(self, encoded) -> list[int]:
        inv_dict = {v: k for k, v in self.dictionary.items()}
        return [inv_dict[encoding] for encoding in encoded]

    def vocab(self):
        return self.dictionary.keys()

encoder = Encoder("tinyshakespeare.txt")
encoded = encoder.encode("Hello World!")
print(encoded)
decoded = encoder.decode(encoded)
print(decoded)

[49, 8, 28, 28, 14, 5, 35, 14, 2, 28, 18, 43]
['H', 'e', 'l', 'l', 'o', ' ', 'W', 'o', 'r', 'l', 'd', '!']


In [3]:
from torch import nn, Tensor, softmax, randn

class Transformer(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.encoder = Encoder("tinyshakespeare.txt")

        vocab_size = len(self.encoder.vocab())
        max_seq_len = 1024
        embed_dim = 256
        hidden_dim = 4 * embed_dim
        num_layers = 16

        self.token_emb = nn.Parameter(randn(vocab_size, embed_dim))
        self.pos_emb = nn.Parameter(randn(max_seq_len, embed_dim))
        self.t_emb = nn.Linear(1, embed_dim)

        self.layers = nn.ModuleList([
            nn.ModuleList([
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim), #W_Q
                nn.Linear(embed_dim, embed_dim), #W_K
                nn.Linear(embed_dim, embed_dim), #W_V

                #FFN
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, embed_dim)])
            for _ in range(num_layers)
        ])

        self.output = nn.Linear(embed_dim, vocab_size)

    def embedding(self, x: Tensor, t: Tensor):
        t_emb = self.t_emb(t)
        return self.token_emb[x] + self.pos_emb[:x.shape[-1]] + t_emb

    def attention(self, Q: Tensor, K: Tensor, V: Tensor):
        d_k = K.shape[-1]
        return softmax(Q @ K.transpose(-2, -1) / (d_k ** 0.5), dim=-1) @ V

    def forward(self, x: Tensor, t: Tensor) -> Tensor:
        x = self.embedding(x, t)
        for ln1, W_Q, W_K, W_V, ln2, linear1, relu, linear2 in self.layers:
            x = x + self.attention(W_Q(ln1(x)), W_K(ln1(x)), W_V(ln1(x)))
            x = x + linear2(relu(linear1(ln2(x))))

        return self.output(x)

In [4]:
import random
import torch
import time
import ipywidgets as widgets
from IPython.display import display
import torch.nn.functional as F

vocab_size = len(encoder.dictionary)

class LogLinear(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.eps = 1e-3

  def forward(self, t):
    t = (1 - self.eps) * t
    alpha_t = 1 - t
    dalpha_t = - (1 - self.eps)
    return dalpha_t, alpha_t
  
noise = LogLinear()

def sample(model, query, length, device, total_steps=20):

    # Widget for printing 
    out = widgets.Output()
    display(out)

    # Tokenize text input
    tokens = model.encoder.encode(query)
    x = torch.randint(0, vocab_size, (1, length,), dtype=torch.long, device=device)
    x[0, :len(tokens)] = torch.tensor(tokens, device=device)
    prompt_mask = torch.arange(x.shape[1], device=device) >= len(tokens)
    
    with torch.no_grad():
        eps = 1e-5
        timesteps = torch.linspace(1, eps, total_steps + 1, device=device)
        for step in range(total_steps):
            
            # Predict probabilites for each token at each position
            t = timesteps[step].view(-1)
            predictions = model.forward(x, t)
            probs = torch.softmax(predictions, dim=-1)

            # Iterate overall all masked tokens
            _, alpha_s = noise(t - ((1 - eps) / total_steps))
            _, alpha_t = noise(t)
            alpha_ts = alpha_t / alpha_s
            d_alpha = alpha_s - alpha_t
            for pos in prompt_mask.nonzero():
                # Equation 4 in https://arxiv.org/pdf/2506.10892
                x_one_hot = F.one_hot(x[0,pos.item()], vocab_size).to(x.dtype).to(x.device)
                posterior = (alpha_t * vocab_size * x_one_hot * probs[0,pos.item()] + (
                    alpha_ts - alpha_t) * probs[0,pos.item()] + d_alpha * x_one_hot + (
                    1 - alpha_ts) * (1 - alpha_s) / vocab_size) / (
                        alpha_t * vocab_size * torch.gather(
                        x_one_hot, -1, x[0,pos.item()][..., None]) + (1 - alpha_t))

                # Sample token from categorical distribution
                x[0,pos.item()] = torch.multinomial(posterior, 1)
            
            # Print out the generated text
            with out:
                out.clear_output(wait=True)
                print(''.join(model.encoder.decode(x[0].tolist())))



## Training

1. Sample chunk of text data: $x_0 \sim TinyShakespeare$
2. Sample masking probability for each token: $t_{token} \sim Uniform(0,1)$
3. Add noise as masking for each token: $x_{t,token}=x_{0,token} (1 - m)$, where $m \sim Bernoulli(t_{tokem})$
4. Run diffusion model $D_{\theta}$ with masked tokens: ${\hat x_0} = D_{\theta}(x_{t},t_{token})$
5. The prediction $\hat x_{0}$ is a categorical distribution for each token. Use cross entropy to train the model for each token: $L = -Σ ~x_{0,token}~log(\hat x_{0,token})$
6. Backprop

In [5]:
import math
import torch
import random
import time
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("tinyshakespeare.txt") as f:
    text = f.read()

split = int(0.9 * len(text))
train_text, val_text = text[:split], text[split:]

seq_len = 128
batch_size = 64
iterations = 100000
checkpoint_every = 10000

transformer = Transformer().to(device)
optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-4)

def grab_chunk(src) -> torch.Tensor:
    start = random.randint(0, len(src) - seq_len)
    return torch.tensor(transformer.encoder.encode(src[start:start + seq_len]), device=device)

def add_noise(x, t):
    mask = (torch.rand_like(x, dtype=torch.float) < t).long()
    uniform_random_tokens = torch.randint(0, vocab_size, x.shape, dtype=torch.long, device=device)
    return x * mask + uniform_random_tokens * (1 - mask), mask

In [6]:
for i in tqdm(range(iterations)):
    batch = torch.stack([grab_chunk(train_text) for _ in range(batch_size)])
    t = random.uniform(0, 1)
    noised_text, _ = add_noise(batch, t)
    preds = transformer(noised_text, torch.tensor([t], device=device))
    loss = torch.nn.functional.cross_entropy(preds.permute(0, 2, 1), batch)
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if i % checkpoint_every == 0:
        sample(transformer, "To be, ", seq_len, device)

# Save final model
sample(transformer, "To be, ", 64, device)
torch.save(transformer.state_dict(), f"udlm.pt")

  0%|          | 0/100000 [00:00<?, ?it/s]

Output()

 10%|█         | 10000/100000 [20:57<3:11:00,  7.85it/s]

Output()

 20%|██        | 20000/100000 [41:57<2:46:07,  8.03it/s]

...

Output()

 90%|█████████ | 90000/100000 [3:04:46<20:25,  8.16it/s]  

Output()

100%|██████████| 100000/100000 [3:25:06<00:00,  8.13it/s]


Output()

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer = Transformer().to(device)
transformer.load_state_dict(torch.load("udlm.pt"))

/tmp/ipykernel_4095187/1406250146.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  transformer.load_state_dict(torch.load("udlm.pt"))


<All keys matched successfully>

In [14]:
sample(transformer, "To be, ", 128, device, 1000)

Output()